In [ ]:
BDD scenario prompt improve

In [ ]:
import re
import os
import json

def parse_evaluation_file(filepath="/kaggle/input/testing2/newSample.txt"):
   
    if not os.path.exists(filepath):
        return None

    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    records = re.split(r'Input Record\d*:', content)
    
    extracted_data = []
    for i, record_text in enumerate(records[1:], start=1):
        
        expected_match = re.search(r'Expected BDD scenario:(.*?)Generated BDD scenario:', record_text, re.DOTALL)
        
        generated_match = re.search(r'Generated BDD scenario:(.*)', record_text, re.DOTALL)

        if expected_match and generated_match:
            expected_output = expected_match.group(1).strip()
            generated_output = generated_match.group(1).strip()
            
            extracted_data.append({
                "id": i,
                "expected": expected_output,
                "generated": generated_output
            })
        else:
            print("")
    
    return extracted_data

if __name__ == "__main__":
    input_filepath = "/kaggle/input/testing2/newSample.txt"
    output_filepath = "evaluation_data.json"

    all_samples = parse_evaluation_file(input_filepath)

    if all_samples:
        with open(output_filepath, 'w', encoding='utf-8') as f:
            json.dump(all_samples, f, indent=4)
        
    else:
        print("")

In [ ]:
Records

In [ ]:
import re
import os
import json 

def parse_evaluation_file(filepath="/kaggle/input/1shotrecordcodet5p-220m/Sample.txt"):
    if not os.path.exists(filepath):
        return None

    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    records = re.split(r'Input BDD Scenario\d+:', content)
    extracted_data = []
    for i, record_text in enumerate(records[1:], start=1):
        expected_match = re.search(r'Expected Record:(.*?)Generated Record:', record_text, re.DOTALL)
        generated_match = re.search(r'Generated Record:(.*)', record_text, re.DOTALL)

        if expected_match and generated_match:
            expected_output = expected_match.group(1).strip()
            generated_output = generated_match.group(1).strip()
            extracted_data.append({
                "id": i,
                "expected": expected_output,
                "generated": generated_output
            })
        else:
            print("")
    
    return extracted_data

if __name__ == "__main__":
    input_filepath = "/kaggle/input/1shotrecordcodet5p-220m/Sample.txt" 
    output_filepath = "evaluation_data.json"
    all_samples = parse_evaluation_file(input_filepath)

    if all_samples:
        with open(output_filepath, 'w', encoding='utf-8') as f:
            json.dump(all_samples, f, indent=4)
    else:
        print("")

Prefix Bdd Scenario and Record

In [ ]:
import re
import os
import json

def parse_new_format_file(filepath="/kaggle/input/prefixrecord/Sample.txt"):
    
    if not os.path.exists(filepath):
        print(f"Error: File not found at '{filepath}'")
        return None

    print(f"--- Starting extraction from '{filepath}' ---")
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    example_blocks = re.split(r'-{11,}\s*Example\s*\d+\s*-{11,}', content)
    
    extracted_data = []
    for i, block_text in enumerate(example_blocks[1:], start=1):
        if not block_text.strip():
            continue

        expected_match = re.search(r'\[CORRECT OUTPUT \(REFERENCE\)\](.*?)\[MODEL GENERATED OUTPUT \(PREDICTION\)\]', block_text, re.DOTALL)
        generated_match = re.search(r'\[MODEL GENERATED OUTPUT \(PREDICTION\)\](.*)', block_text, re.DOTALL)

        if expected_match and generated_match:
            expected_output = expected_match.group(1).strip()
            generated_output = generated_match.group(1).strip()
            
            if '-----------' in generated_output:
                generated_output = re.split(r'-{11,}', generated_output)[0].strip()
            
            extracted_data.append({
                "id": i,
                "expected": expected_output,
                "generated": generated_output
            })
        else:
            print("")

    return extracted_data

if __name__ == "__main__":
    input_filepath = "/kaggle/input/prefixrecord/Sample.txt" 
    
    output_filepath = "evaluation_data.json"

    all_samples = parse_new_format_file(input_filepath)

    if all_samples:
        with open(output_filepath, 'w', encoding='utf-8') as f:
            json.dump(all_samples, f, indent=4)
        
    else:
        print("")

In [ ]:
!pip install cerebras-cloud-sdk

In [ ]:
import os
import json
import re
import time
from cerebras.cloud.sdk import Cerebras
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
cerebras_api_key = user_secrets.get_secret("Cerebras") 
client = Cerebras(api_key=cerebras_api_key)


INPUT_JSON_PATH = "/kaggle/input/prefixrecord/evaluation_data.json" 
MODEL_ID = "gpt-oss-120b" 
BATCH_SIZE = 10 

SYSTEM_PROMPT = """You are an expert AI judge. You will be given a BATCH of sample pairs.
For EACH pair, compare the 'Generated' text to the 'Expected' text.

Use this 4-point rubric:
1 (Poor): Irrelevant or completely incorrect.
2 (Partial): Misses significant info or has major structural errors.
3 (Good): Captures primary intent with minor stylistic differences.
4 (Excellent): Perfect match.

Your response MUST be a JSON array of objects, one for each sample in the batch, following this exact order:
[
  {"sample_id": X, "score": 1-4, "reasoning": "..."}
]"""

def get_batch_judgments(api_client, batch):
    batch_text = ""
    for sample in batch:
        batch_text += f"--- SAMPLE ID: {sample['id']} ---\n"
        batch_text += f"EXPECTED:\n{sample['expected']}\n"
        batch_text += f"GENERATED:\n{sample['generated']}\n\n"

    try:
        response = api_client.chat.completions.create(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": batch_text},
            ],
            temperature=0, 
        )
        raw_text = response.choices[0].message.content
        
        json_match = re.search(r'\[.*\]', raw_text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
        return {"error": "JSON array not found", "raw": raw_text}
    except Exception as e:
        return {"error": str(e)}

with open(INPUT_JSON_PATH, 'r') as f:
    all_samples = json.load(f)


final_results = []
for i in range(0, len(all_samples), BATCH_SIZE):
    batch = all_samples[i : i + BATCH_SIZE]
    print(f"Evaluating batch starting at index {i}...")
    
    batch_results = get_batch_judgments(client, batch)
    
    if isinstance(batch_results, list):
        final_results.extend(batch_results)
    else:
        print("")
    
    time.sleep(6.5)

with open("gpt_batch_results.json", 'w') as f:
    json.dump(final_results, f, indent=4)


In [ ]:
rm /kaggle/working/*